# 파일 파이프라인 추출 모듈 점검

합성 입력만 사용해 각 모듈의 대표 함수를 호출합니다.

In [ ]:
from pathlib import Path
import json
import sys
import tempfile

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
for extra in (REPO / "src", REPO / "src" / "card_benefit"):
    if str(extra) not in sys.path:
        sys.path.insert(0, str(extra))
SCRATCH_ROOT = Path(tempfile.mkdtemp())


## 청크 이어하기와 호출 설정

In [ ]:
import os
from unittest.mock import patch
from payment_card.file_pipeline.extraction.batch import run_batch
from payment_card.file_pipeline.extraction.llm_adapter import create_llm_call

class SyntheticClient:
    def __init__(self, **kwargs):
        pass

    def invoke(self, prompt):
        return json.dumps({"schema_ver": "ONTOLOGY_V1", "benefits": [
            {"id": "b1", "name": "합성 할인", "evidence": [{"text": "합성 할인"}]}
        ]}, ensure_ascii=False)

synthetic_dir = SCRATCH_ROOT / "batch_resume"
synthetic_dir.mkdir(parents=True, exist_ok=True)
chunk_file = synthetic_dir / "synthetic.chunks.json"
chunk_file.write_text(json.dumps({
    "doc_id": "synthetic", "doc_sha256": "a" * 64,
    "chunks": [{"chunk_id": "synthetic-c1", "page_no": 1, "start_pos": 1, "text": "합성 할인"}],
}, ensure_ascii=False), encoding="utf-8")
with patch.dict(os.environ, {"FP_NOTEBOOK_SYNTHETIC_KEY": "synthetic"}):
    synthetic_call = create_llm_call("synthetic-model", "FP_NOTEBOOK_SYNTHETIC_KEY", client_factory=SyntheticClient)
context = {"provider": "synthetic", "model": "synthetic-model", "temperature": None}
result_dir = synthetic_dir / "results"
first = run_batch(chunk_file, synthetic_call, result_dir, resume_context=context)
second = run_batch(chunk_file, synthetic_call, result_dir, max_chunks=0, resume_context=context)
print(json.dumps({"first_executed": first["documents"][0]["summary"]["executed_chunks"],
                  "resumed": second["documents"][0]["summary"]["skipped"]}, ensure_ascii=False))

import fcntl
with (result_dir / ".file_pipeline.lock").open("a+") as lock:
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    try:
        run_batch(chunk_file, synthetic_call, result_dir, resume_context=context)
    except RuntimeError as error:
        print(type(error).__name__)

# 저장된 결과 내용이 달라지면 기존 파일을 보존하며 이어하기를 중단합니다.
result_path = result_dir / "synthetic-c1.result.json"
original = result_path.read_bytes()
try:
    changed = json.loads(original)
    changed["extraction"] = {}
    result_path.write_text(json.dumps(changed), encoding="utf-8")
    try:
        run_batch(chunk_file, synthetic_call, result_dir, resume_context=context)
    except ValueError as error:
        print(type(error).__name__)
finally:
    result_path.write_bytes(original)


## 파일럿 재실행 전 출력 확인

합성 작업 폴더에 그래프 결과가 있으면 모델 호출을 준비하기 전에 중단합니다.


In [ ]:
from payment_card.file_pipeline.extraction.pilot_run import run_pilot

pilot_work = SCRATCH_ROOT / "pilot_existing"
pilot_graph = pilot_work / "graph"
pilot_graph.mkdir(parents=True, exist_ok=True)
marker = pilot_graph / "existing.txt"
marker.write_text("preserved", encoding="utf-8")
try:
    run_pilot(SCRATCH_ROOT / "unused.csv", SCRATCH_ROOT / "unused_chunks", pilot_work,
              1, "synthetic", execute=True)
except FileExistsError:
    print("기존 결과 보존:", marker.read_text(encoding="utf-8") == "preserved")
